In [ ]:
from pathlib import Path
import pickle
import colorsys

import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from topostats.damage.io import construct_grains_collection_from_topostats_files
from topostats.damage.classes import UnanalysedGrainCollection, GrainCollection
from topostats.damage.damage_processing import (
    find_defects_in_height_and_curvature,
    calculate_turn_in_distance,
    smooth_height_traces,
)

# Config

In [ ]:
from topostats.measure.curvature import total_turn_in_region_radians

dir_base = Path("/Volumes/shared/pyne_group/Shared/AFM_Data/dna_damage/Cs137_irradiations")
dir_remote_analysis = dir_base / "20260204-analysis-getting-back-into-the-project"
threshold_contour_length = 250  # Cutoff for the minimum contour length for grains to consider
dir_results = dir_remote_analysis / "analysis_results"

local_analysis: bool = True  # Whether to do the analysis using a locally saved cache file.
force_reload_from_file: bool = False  # Whether to force-reload the data from the original topostats files
dir_local_pkl_cache = Path("/Users/sylvi/topo_data/dna_damage_cache")  # Where to load/save the local cache file

if local_analysis:
    dir_results = dir_local_pkl_cache / "analysis_results"  # Where to save the results of the analysis
else:
    dir_results = dir_remote_analysis / "output" / "analysis_results"  # Where to save the results of the analysis

# height trace smoothing window size nm
height_trace_smoothing_window_length_nm = 10

# defects
height_defect_method = "iqr"
height_threshold_iqr_multiplier = 3.0
height_threshold_absolute_nm = 0.8
height_threshold_percentage_of_median = 0.8

curvature_defect_method = "turn_in_distance"
curvature_threshold_iqr_multiplier = 3.0
curvature_threshold_absolute_pernm = 0.1
curvature_turn_in_distance_turn_threshold_deg = 150

connect_close_defect_threshold_nm = 10
coinciding_defect_threshold_nm = 10
turn_in_distance_window_length_nm = 10
turn_in_distance_window_end_sampling_points = 3

# plotting
folder_to_labels = {
    "Controls": "Control",
    "MilliQ/5_percent_damage": "MilliQ 5%",
    "MilliQ/20_percent_damage": "MilliQ 20%",
    "MilliQ/50_percent_damage": "MilliQ 50%",
    "TE/5_percent_damage": "TE 5%",
    "TE/20_percent_damage": "TE 20%",
    "TE/50_percent_damage": "TE 50%",
}
folder_to_colours_hsv = {
    "Controls": (0 / 360, 0 / 100, 35 / 100),
    "MilliQ/5_percent_damage": (0 / 360, 40 / 100, 40 / 100),
    "MilliQ/20_percent_damage": (0 / 360, 60 / 100, 60 / 100),
    "MilliQ/50_percent_damage": (0 / 360, 80 / 100, 80 / 100),
    "TE/5_percent_damage": (183 / 360, 40 / 100, 40 / 100),
    "TE/20_percent_damage": (183 / 360, 60 / 100, 60 / 100),
    "TE/50_percent_damage": (183 / 360, 80 / 100, 80 / 100),
}
folder_to_colours_rgb = {folder: colorsys.hsv_to_rgb(*hsv) for folder, hsv in folder_to_colours_hsv.items()}
plotting_folder_order = [
    "Controls",
    "MilliQ/5_percent_damage",
    "MilliQ/20_percent_damage",
    "MilliQ/50_percent_damage",
    "TE/5_percent_damage",
    "TE/20_percent_damage",
    "TE/50_percent_damage",
]

In [ ]:
def update_x_labels(ax: plt.Axes):
    # Update x-axis folder labels to be more readable
    x_labels = [item.get_text() for item in ax.get_xticklabels()]
    new_labels = [folder_to_labels.get(label, label) for label in x_labels]
    ax.set_xticklabels(new_labels, rotation=45, ha="right")

# Loading data

In [ ]:
assert dir_local_pkl_cache.exists()
dir_results.mkdir(exist_ok=True)
assert dir_results.exists()
if not local_analysis:
    assert dir_base.exists()
    assert dir_remote_analysis.exists()
    dir_processed_data = dir_remote_analysis / "output"
    assert dir_processed_data.exists()

    topo_files = list(dir_processed_data.glob("*/**/processed/*.topostats"))
    print(f"found {len(topo_files)} topo files")

    # Load the corresponding csv file
    csv_grain_stats: Path = dir_processed_data / "grain_statistics.csv"
    assert csv_grain_stats.exists()
    df_grain_stats = pd.read_csv(csv_grain_stats)
    print(f"grain stats columns: {df_grain_stats.columns}")

    # convert some columns to nanometres
    df_grain_stats["total_contour_length"] /= 1e-9

    # convert basename to only use the parent directory
    df_grain_stats["basename"] = df_grain_stats["basename"].apply(lambda x: Path(x).parent)

    # Strip the file extension from the "image" column
    df_grain_stats["image"] = df_grain_stats["image"].apply(lambda x: Path(x).stem)

    # replace NaN in "num_crossings" with 0
    df_grain_stats["num_crossings"] = df_grain_stats["num_crossings"].fillna(0)

    # drop any row where the basename contains "diabolical tips"
    df_grain_stats = df_grain_stats[~df_grain_stats["basename"].astype(str).str.contains("diabolical tips")]

    print(df_grain_stats["basename"].value_counts())

# Sample vetting

In [ ]:
if not local_analysis:
    # plot contour length distributions
    sns.stripplot(data=df_grain_stats, x="basename", y="total_contour_length", s=2)
    sns.violinplot(data=df_grain_stats, x="basename", y="total_contour_length", inner=None)
    plt.xticks(rotation=90)
    plt.title("Contour length distributions")
    plt.show()

    # drop any rows with contour length less than a threshold
    n_rows_before = len(df_grain_stats)
    df_grain_stats = df_grain_stats[df_grain_stats["total_contour_length"] >= threshold_contour_length]
    n_rows_after = len(df_grain_stats)
    print(
        f"dropped {n_rows_before - n_rows_after} rows with contour length < {threshold_contour_length} nm."
        f"remaining rows: {n_rows_after}"
    )

    sns.stripplot(data=df_grain_stats, x="basename", y="total_contour_length", s=2)
    sns.violinplot(data=df_grain_stats, x="basename", y="total_contour_length", inner=None)
    plt.xticks(rotation=90)
    plt.title("Contour length distributions")
    plt.show()

# Construct initial graincollection

In [ ]:
if not local_analysis:
    unanalysed_grain_collection: UnanalysedGrainCollection = construct_grains_collection_from_topostats_files(
        dir_to_save_cache_files=dir_results,
        dir_processed_data=dir_processed_data,
        df_grain_stats=df_grain_stats,
        bbox_padding=1,
        force_reload=force_reload_from_file,
    )
    print(unanalysed_grain_collection)

    # Save the object to a pickle file
    pkl_file = dir_local_pkl_cache / "unanalysed_grain_collection.pkl"
    with open(pkl_file, "wb") as f:
        pickle.dump(unanalysed_grain_collection, f)
    # save a cache of the csv file too
    df_grain_stats.to_csv(dir_local_pkl_cache / "grain_statistics.csv", index=False)
else:
    # Load the object from the pickle file
    pkl_file = dir_local_pkl_cache / "unanalysed_grain_collection.pkl"
    with open(pkl_file, "rb") as f:
        unanalysed_grain_collection: UnanalysedGrainCollection = pickle.load(f)
    print(unanalysed_grain_collection)
    # load the cached csv file too
    df_grain_stats = pd.read_csv(dir_local_pkl_cache / "grain_statistics.csv")

grains_to_plot = [1, 5, 22, 23, 27, 80, 97, 105]
for grain_id in grains_to_plot:
    grain = unanalysed_grain_collection.unanalysed_grains[grain_id]
    grain.plot()

# Create new analysed data model

In [ ]:
grain_collection: GrainCollection = GrainCollection.from_unanalysed_grain_collection(
    unanalysed_collection=unanalysed_grain_collection,
    coinciding_defect_threshold_nm=coinciding_defect_threshold_nm,
)

print(grain_collection)

In [ ]:
# Smooth height traces
bad_grain = smooth_height_traces(
    grain_collection=grain_collection,
    window_length_nm=height_trace_smoothing_window_length_nm,
)

print(grain_collection)

example_grain = grain_collection.get_random_grain(seed=0)
height_trace = example_grain.molecule_data_collection.molecules[0].spline_coords_heights
smoothed_height_trace = example_grain.molecule_data_collection.molecules[0].smoothed_spline_coords_heights
point_spacing_nm = example_grain.molecule_data_collection.molecules[0].point_spacing_nm
x_positions_nm = np.arange(len(height_trace)) * point_spacing_nm
assert smoothed_height_trace is not None
plt.plot(x_positions_nm, height_trace, label="Original")
plt.plot(x_positions_nm, smoothed_height_trace, label="Smoothed")
plt.legend()
plt.title(
    f"Example height trace before and after smoothing. Window length {height_trace_smoothing_window_length_nm} nm"
)
plt.ylabel("Height (nm)")
plt.xlabel("Position along molecule (nm)")
plt.show()

In [ ]:
bad_grains = calculate_turn_in_distance(
    grain_collection=grain_collection,
    turn_in_distance_window_length_nm=turn_in_distance_window_length_nm,
    turn_in_distance_window_end_sampling_points=turn_in_distance_window_end_sampling_points,
)

print(f"encoutnered {len(bad_grains)} bad grains")

# curvature vs height analysis

In [ ]:
all_turn_in_distance_heights = []
for grain_index, grain in grain_collection.grains.items():
    for molecule_index, molecule in grain.molecule_data_collection.molecules.items():
        if molecule.curvature_data is not None:
            if (
                molecule.curvature_data.turn_in_distances_deg is not None
                and molecule.smoothed_spline_coords_heights is not None
            ):
                for turn_in_distance, height, curvature in zip(
                    molecule.curvature_data.turn_in_distances_deg,
                    molecule.smoothed_spline_coords_heights,
                    molecule.curvature_data.curvatures,
                ):
                    all_turn_in_distance_heights.append(
                        {
                            "turn_in_distance": np.abs(turn_in_distance),
                            "height": height,
                            "curvature": np.abs(curvature),
                            "folder": grain.folder,
                        }
                    )
df_turn_in_distance_heights = pd.DataFrame(all_turn_in_distance_heights)
max_height = df_turn_in_distance_heights["height"].max()


# plot a grid of folders for turn in distance and height
max_turn_in_distance = df_turn_in_distance_heights["turn_in_distance"].max()
fig, ax = plt.subplots(
    (len(plotting_folder_order) + 1) // 2, 2, figsize=(12, 6 * ((len(plotting_folder_order) + 1) // 2))
)
ax = ax.flatten()
for i, folder in enumerate(plotting_folder_order):
    folder_data = df_turn_in_distance_heights[df_turn_in_distance_heights["folder"] == folder]
    folder_colour = folder_to_colours_rgb[folder]
    sns.scatterplot(data=folder_data, x="turn_in_distance", y="height", alpha=0.5, ax=ax[i], color=folder_colour)
    ax[i].set_title(f"Turn in distance vs smoothed height for {folder_to_labels.get(folder, folder)}")
    ax[i].set_xlabel(f"Turn in distance (deg) for window length {turn_in_distance_window_length_nm} nm")
    ax[i].set_ylabel(f"Smoothed height (nm) with window length {height_trace_smoothing_window_length_nm} nm")
    ax[i].set_xlim(0, max_turn_in_distance)
    ax[i].set_ylim(0, max_height)
plt.tight_layout()
plt.show()

# plot a grid of folders for curvature and height
max_curvature = df_turn_in_distance_heights["curvature"].max()
fig, ax = plt.subplots(
    (len(plotting_folder_order) + 1) // 2, 2, figsize=(12, 6 * ((len(plotting_folder_order) + 1) // 2))
)
ax = ax.flatten()
for i, folder in enumerate(plotting_folder_order):
    folder_data = df_turn_in_distance_heights[df_turn_in_distance_heights["folder"] == folder]
    folder_colour = folder_to_colours_rgb[folder]
    sns.scatterplot(data=folder_data, x="curvature", y="height", alpha=0.5, ax=ax[i], color=folder_colour)
    ax[i].set_title(f"Curvature vs smoothed height for {folder_to_labels.get(folder, folder)}")
    ax[i].set_xlabel(f"Smoothed curvature (discrete points) (radians/nm)")
    ax[i].set_ylabel(f"Smoothed height (nm) with window length {height_trace_smoothing_window_length_nm} nm")
    ax[i].set_xlim(0, max_curvature)
    ax[i].set_ylim(0, max_height)
plt.tight_layout()
plt.show()

# Plot a grid of folders for turn in distance and curvature
fig, ax = plt.subplots(
    (len(plotting_folder_order) + 1) // 2, 2, figsize=(12, 6 * ((len(plotting_folder_order) + 1) // 2))
)
ax = ax.flatten()
for i, folder in enumerate(plotting_folder_order):
    folder_data = df_turn_in_distance_heights[df_turn_in_distance_heights["folder"] == folder]
    folder_colour = folder_to_colours_rgb[folder]
    sns.scatterplot(data=folder_data, x="turn_in_distance", y="curvature", alpha=0.5, ax=ax[i], color=folder_colour)
    ax[i].set_title(f"Turn in distance vs curvature for {folder_to_labels.get(folder, folder)}")
    ax[i].set_xlabel(f"Turn in distance (deg) for window length {turn_in_distance_window_length_nm} nm")
    ax[i].set_ylabel(f"Smoothed curvature (discrete points) (radians/nm)")
    ax[i].set_xlim(0, max_turn_in_distance)
    ax[i].set_ylim(0, max_curvature)
plt.tight_layout()
plt.show()


# plot the distributions of curvatures for each sample type
fig, ax = plt.subplots(figsize=(10, 6))
for folder in plotting_folder_order:
    folder_data = df_turn_in_distance_heights[df_turn_in_distance_heights["folder"] == folder]
    folder_colour = folder_to_colours_rgb[folder]
    sns.kdeplot(data=folder_data, x="curvature", ax=ax, label=folder_to_labels.get(folder, folder), color=folder_colour)
ax.set_title("Distribution of smoothed curvatures for each sample type")
ax.set_xlabel(f"Smoothed curvature (discrete points) (radians/nm)")
ax.set_ylabel("Density")
ax.legend()
plt.show()

# plot the distributions of turn in distances for each sample type
fig, ax = plt.subplots(figsize=(10, 6))
for folder in plotting_folder_order:
    folder_data = df_turn_in_distance_heights[df_turn_in_distance_heights["folder"] == folder]
    folder_colour = folder_to_colours_rgb[folder]
    sns.kdeplot(
        data=folder_data, x="turn_in_distance", ax=ax, label=folder_to_labels.get(folder, folder), color=folder_colour
    )
ax.set_title(
    f"Distribution of turn in distances for each sample type. Window length {turn_in_distance_window_length_nm} nm"
)
ax.set_xlabel(f"Turn in distance (deg)")
ax.set_ylabel("Density")
ax.legend()
plt.show()

# Defect detection

In [ ]:
bad_grains = find_defects_in_height_and_curvature(
    grain_collection=grain_collection,
    height_defect_method=height_defect_method,
    height_threshold_iqr_multiplier=height_threshold_iqr_multiplier,
    height_threshold_absolute_nm=height_threshold_absolute_nm,
    height_threshold_percentage_of_median=height_threshold_percentage_of_median,
    curvature_defect_method=curvature_defect_method,
    curvature_threshold_iqr_multiplier=curvature_threshold_iqr_multiplier,
    curvature_threshold_absolute_pernm=curvature_threshold_absolute_pernm,
    curvature_turn_in_distance_turn_threshold_deg=curvature_turn_in_distance_turn_threshold_deg,
    connect_close_defect_threshold_nm=connect_close_defect_threshold_nm,
)

# remove bad grains
grain_collection.remove_grains(bad_grains)

print(grain_collection)

# Plot defect quantification

In [ ]:
# dataframe with columns of grain_id, folder, num_height_defects, num_curvature_defects
height_curvature_defect_counts = []
for grain_id, grain in grain_collection.grains.items():
    folder = grain.folder
    num_height_defects = grain.num_height_defects
    num_curvature_defects = grain.num_curvature_defects
    height_curvature_defect_counts.append(
        {
            "grain_id": grain_id,
            "folder": folder,
            "num_height_defects": num_height_defects,
            "num_curvature_defects": num_curvature_defects,
            "image": grain.filename,
            "grain_number": grain.file_grain_id,
            "coinciding_defects": grain.num_coinciding_defects,
        }
    )
df_defect_counts = pd.DataFrame(height_curvature_defect_counts)

# Violin plot
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
sns.violinplot(
    data=df_defect_counts, x="folder", y="num_height_defects", inner=None, ax=ax[0], order=plotting_folder_order
)
sns.stripplot(
    data=df_defect_counts, x="folder", y="num_height_defects", color="k", size=2, ax=ax[0], order=plotting_folder_order
)
sns.violinplot(
    data=df_defect_counts, x="folder", y="num_curvature_defects", inner=None, ax=ax[1], order=plotting_folder_order
)
sns.stripplot(
    data=df_defect_counts,
    x="folder",
    y="num_curvature_defects",
    color="k",
    size=2,
    ax=ax[1],
    order=plotting_folder_order,
)
update_x_labels(ax[0])
update_x_labels(ax[1])
ax[0].set_title(f"Height defects per grain")
ax[1].set_title(f"Curvature defects per grain")
plt.tight_layout()
plt.show()

sns.violinplot(data=df_defect_counts, x="folder", y="coinciding_defects", inner=None, order=plotting_folder_order)
sns.stripplot(data=df_defect_counts, x="folder", y="coinciding_defects", color="k", size=2, order=plotting_folder_order)
update_x_labels(plt.gca())
plt.ylabel("Number of coinciding defects")
plt.show()

print(
    f"whether grains have coinciding defect per folder value counts for threshold {coinciding_defect_threshold_nm} nm:"
)
total_coinciding_defects = df_defect_counts["coinciding_defects"].apply(lambda x: bool(x > 0)).value_counts()
total_proportion_with_coinciding_defects = total_coinciding_defects.get(True, 0) / len(df_defect_counts)
print(
    f"Total: {total_coinciding_defects.get(True, 0)} out of {len(df_defect_counts)} grains ({total_proportion_with_coinciding_defects:.2%}) with coinciding defects"
)
for folder in plotting_folder_order:
    folder_df = df_defect_counts[df_defect_counts["folder"] == folder]
    coinciding_defects = 0
    for count in folder_df["coinciding_defects"]:
        coinciding_defects += bool(count > 0)
    proportion_with_coinciding_defects = coinciding_defects / len(folder_df)
    print(f"{folder:<25}: {coinciding_defects:<5} {proportion_with_coinciding_defects:<10.2%} with coinciding defects")


sampled_grains = grain_collection.sample(n=1, seed=1)
for grain in sampled_grains.grains.values():
    grain.plot(
        mask_alpha=0.0,
        curvature_defects=False,
        height_defects=False,
        coinciding_defects=False,
        linemode="curvature",
        curvature_absolute=True,
        linewidth=3.0,
    )

In [ ]:
print("indexes of grains with coinciding defects:")
for grain_id, grain in grain_collection.grains.items():
    if grain.num_coinciding_defects > 0:
        print(f"{grain_id}")


sample_grains = [9, 28, 76, 111, 123, 162]
for grain_id in sample_grains:
    grain = grain_collection.grains[grain_id]
    grain.plot(
        mask_alpha=0.1, curvature_defects=True, height_defects=True, coinciding_defects=True, linemode="curvature"
    )

In [ ]:
# Visualise turn in distance

sampled_grains = grain_collection.sample(n=1, seed=1)
for grain in sampled_grains.grains.values():
    grain.plot(
        mask_alpha=0.0,
        curvature_defects=False,
        height_defects=False,
        coinciding_defects=False,
        linemode="turn_in_distance",
        turn_in_distance_absolute=True,
        turn_in_distance_display_value_interval=20,
        linewidth=3.0,
        figsize=(10, 10),
    )

In [ ]:
grain_model = grain_collection.grains[0]
import numpy as np

points_nm = grain_model.molecule_data_collection.molecules[0].spline_coords_nm
circular = grain_model.molecule_data_collection.molecules[0].circular
distances_nm = []
for index in range(len(points_nm)):
    if index == 0:
        if not circular:
            distances_nm.append(0.0)
            continue
    point = points_nm[index]
    previous_point = points_nm[index - 1]
    distance = np.linalg.norm(point - previous_point)
    distances_nm.append(distance)
print(distances_nm)

In [ ]:
sample_grains = grain_collection.sample(n=5, seed=0)

for grain in sample_grains.grains.values():
    grain.plot(
        linemode="spline",
    )

# Merge stats into grainstats

In [ ]:
# Create a copy of the grain_statistics dataframe
df_grain_stats_copy = df_grain_stats.copy()

df_grain_stats_merged = df_grain_stats_copy.merge(df_defect_counts, on=["image", "grain_number"])

# save to csv
df_grain_stats_merged.to_csv(dir_results / "defect_grain_statistics.csv", index=False)